# Exploratory Data Analysis — Edge AI Acoustic Border Intrusion Detection Dataset

**Paper Title (Draft):** *Edge AI-Based Acoustic Footstep Detection for Border Surveillance Using TinyML and LoRa*

---

### Notebook Objectives
This EDA notebook systematically characterises the `BorderIntrusionDetection-Data` acoustic corpus
prior to TinyML model development. The analysis covers:

| Section | Focus |
|---------|-------|
| §1 | Dataset inventory & file integrity |
| §2 | Class distribution & duration statistics |
| §3 | Waveform morphology per class |
| §4 | Spectral analysis (STFT, Mel, CQT) |
| §5 | MFCC feature characterisation |
| §6 | Handcrafted feature extraction & statistics |
| §7 | Inter-class feature separability (PCA, t-SNE) |
| §8 | Correlation & redundancy analysis |
| §9 | Dataset quality & edge case detection |

**Dataset Path:** `/kaggle/input/datasets/katakuricharlotte/borderintrusiondetection-data/dataset`

In [ ]:
import os, glob, warnings, itertools
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

import librosa
import librosa.display
import soundfile as sf

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from scipy.stats import kurtosis, skew
from scipy.signal import welch

warnings.filterwarnings("ignore")

# ── Research Plot Style ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         12,
    "axes.labelsize":    13,
    "axes.titlesize":    13,
    "axes.linewidth":    1.2,
    "xtick.labelsize":   11,
    "ytick.labelsize":   11,
    "xtick.major.size":  6,
    "ytick.major.size":  6,
    "xtick.minor.size":  3,
    "ytick.minor.size":  3,
    "legend.fontsize":   11,
    "legend.frameon":    True,
    "legend.edgecolor":  "0.4",
    "grid.linestyle":    ":",
    "grid.linewidth":    0.7,
    "grid.alpha":        0.85,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for sp in ax.spines.values():
        sp.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=True, right=True)

CLASS_COLORS = {
    "footsteps": "#2166ac",
    "noise":     "#d6604d",
    "balastic":  "#4dac26",
}
CLASS_MARKERS = {"footsteps": "o", "noise": "s", "balastic": "^"}

DATASET_PATH = Path("/kaggle/input/datasets/katakuricharlotte/borderintrusiondetection-data/dataset")
SR_TARGET    = 16000   # resample target for TinyML pipeline
N_MFCC       = 13
N_FFT        = 512
HOP_LENGTH   = 256
N_MELS       = 40

os.makedirs("figures", exist_ok=True)
print("Classes found:", [p.name for p in sorted(DATASET_PATH.iterdir()) if p.is_dir()])

## §1 — Dataset Inventory & File Integrity

Scan all `.wav` files, extract metadata (duration, sample rate, channels), and
flag any corrupted or unreadable files before proceeding.

In [ ]:
records = []
corrupt = []

for cls_dir in sorted(DATASET_PATH.iterdir()):
    if not cls_dir.is_dir():
        continue
    label = cls_dir.name
    for wav_path in sorted(cls_dir.rglob("*.wav")):
        try:
            info = sf.info(str(wav_path))
            records.append({
                "path":       str(wav_path),
                "filename":   wav_path.name,
                "label":      label,
                "duration_s": round(info.duration, 4),
                "samplerate": info.samplerate,
                "channels":   info.channels,
                "frames":     info.frames,
                "subtype":    info.subtype,
            })
        except Exception as e:
            corrupt.append({"path": str(wav_path), "error": str(e)})

df = pd.DataFrame(records)

print("=" * 55)
print(f"  Total audio files  : {len(df)}")
print(f"  Corrupt / unreadable: {len(corrupt)}")
print(f"  Classes            : {df['label'].unique().tolist()}")
print(f"  Unique sample rates: {df['samplerate'].unique().tolist()}")
print(f"  Unique channels    : {df['channels'].unique().tolist()}")
print("=" * 55)

display(df.groupby("label").agg(
    count=("filename","count"),
    total_duration_s=("duration_s","sum"),
    mean_duration_s=("duration_s","mean"),
    min_duration_s=("duration_s","min"),
    max_duration_s=("duration_s","max"),
    std_duration_s=("duration_s","std"),
).round(3))

if corrupt:
    print("\n⚠ Corrupt files:")
    for c in corrupt:
        print(" ", c)

## §2 — Class Distribution & Duration Analysis

An unbalanced dataset causes biased classifiers. Here we visualise:
- Sample count per class
- Total and per-sample audio duration
- Duration distribution (violin + box overlay)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# ── A: Sample count bar ──────────────────────────────────────────────────────
ax = axes[0]
counts = df["label"].value_counts().reindex(CLASS_COLORS.keys()).dropna()
bars = ax.bar(counts.index, counts.values,
              color=[CLASS_COLORS[c] for c in counts.index],
              edgecolor="black", linewidth=0.8, width=0.55, zorder=3)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            str(int(val)), ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_xlabel("Class")
ax.set_ylabel("Number of Samples")
ax.set_title("(a) Sample Count per Class")
paper_axes(ax)

# ── B: Total duration bar ────────────────────────────────────────────────────
ax = axes[1]
total_dur = df.groupby("label")["duration_s"].sum().reindex(CLASS_COLORS.keys()).dropna() / 60
bars = ax.bar(total_dur.index, total_dur.values,
              color=[CLASS_COLORS[c] for c in total_dur.index],
              edgecolor="black", linewidth=0.8, width=0.55, zorder=3)
for bar, val in zip(bars, total_dur.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}m", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_xlabel("Class")
ax.set_ylabel("Total Duration (minutes)")
ax.set_title("(b) Total Audio Duration per Class")
paper_axes(ax)

# ── C: Duration distribution violin ─────────────────────────────────────────
ax = axes[2]
classes_ordered = [c for c in CLASS_COLORS if c in df["label"].unique()]
data_violin = [df[df["label"] == c]["duration_s"].values for c in classes_ordered]
vp = ax.violinplot(data_violin, positions=range(len(classes_ordered)),
                   showmedians=True, showextrema=True, widths=0.6)
for i, (body, cls) in enumerate(zip(vp["bodies"], classes_ordered)):
    body.set_facecolor(CLASS_COLORS[cls])
    body.set_alpha(0.65)
    body.set_edgecolor("black")
    body.set_linewidth(0.8)
vp["cmedians"].set_color("black")
vp["cmedians"].set_linewidth(1.5)
vp["cbars"].set_linewidth(0.8)
vp["cmins"].set_linewidth(0.8)
vp["cmaxes"].set_linewidth(0.8)

# Overlay box plots
bp = ax.boxplot(data_violin, positions=range(len(classes_ordered)),
                widths=0.12, patch_artist=True,
                medianprops=dict(color="white", linewidth=1.5),
                whiskerprops=dict(linewidth=0.8),
                capprops=dict(linewidth=0.8),
                flierprops=dict(marker="x", markersize=4, markeredgewidth=0.8))
for patch in bp["boxes"]:
    patch.set_facecolor("black")
    patch.set_alpha(0.6)

ax.set_xticks(range(len(classes_ordered)))
ax.set_xticklabels(classes_ordered)
ax.set_xlabel("Class")
ax.set_ylabel("Duration (seconds)")
ax.set_title("(c) Duration Distribution")
paper_axes(ax)

plt.tight_layout()
plt.savefig("figures/fig1_class_distribution.pdf")
plt.show()
print("Saved → figures/fig1_class_distribution.pdf")

## §3 — Waveform Morphology Analysis

Temporal amplitude envelopes reveal characteristic patterns:
- **Footsteps**: rhythmic transient bursts with clear attack/decay
- **Ballistic**: sharp high-amplitude impulse followed by silence
- **Noise**: quasi-stationary stochastic signal

One representative sample per class is analysed with its RMS energy envelope overlaid.

In [ ]:
fig, axes = plt.subplots(len(CLASS_COLORS), 2, figsize=(14, 9),
                         gridspec_kw={"width_ratios": [3, 1]})

for row, (label, color) in enumerate(CLASS_COLORS.items()):
    subset = df[df["label"] == label]
    if subset.empty:
        continue
    # pick median-duration sample as representative
    med_dur = subset["duration_s"].median()
    path = subset.iloc[(subset["duration_s"] - med_dur).abs().argsort().iloc[0]]["path"]

    y, sr = librosa.load(path, sr=SR_TARGET, mono=True)
    t = np.linspace(0, len(y) / sr, len(y))

    # RMS envelope
    frame_len = 512
    rms = librosa.feature.rms(y=y, frame_length=frame_len, hop_length=HOP_LENGTH)[0]
    t_rms = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=HOP_LENGTH)

    # ── Waveform ─────────────────────────────────────────────────────────────
    ax = axes[row, 0]
    ax.fill_between(t, y, alpha=0.45, color=color, linewidth=0)
    ax.plot(t, y, color=color, linewidth=0.4, alpha=0.7)
    ax.plot(t_rms, rms, color="black", linewidth=1.4, label="RMS envelope")
    ax.plot(t_rms, -rms, color="black", linewidth=1.4)
    ax.set_xlim(0, t[-1])
    ax.set_ylim(-1.05, 1.05)
    ax.set_ylabel("Amplitude")
    ax.set_title(f"({chr(97+row*2)}) Waveform — {label.capitalize()}  [{Path(path).name}]")
    ax.legend(loc="upper right")
    paper_axes(ax)

    # ── Amplitude PDF histogram ───────────────────────────────────────────────
    ax2 = axes[row, 1]
    ax2.hist(y, bins=80, orientation="horizontal", color=color,
             edgecolor="none", alpha=0.75, density=True)
    ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax2.set_xlabel("Density")
    ax2.set_ylabel("Amplitude")
    ax2.set_title(f"({chr(97+row*2+1)}) Amplitude PDF")
    ax2.text(0.97, 0.97, f"Kurt={kurtosis(y):.2f}\nSkew={skew(y):.2f}",
             transform=ax2.transAxes, ha="right", va="top", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="0.4"))
    paper_axes(ax2)

axes[-1, 0].set_xlabel("Time (s)")
plt.tight_layout()
plt.savefig("figures/fig2_waveforms.pdf")
plt.show()
print("Saved → figures/fig2_waveforms.pdf")

## §4 — Spectral Analysis

### 4.1 Short-Time Fourier Transform (STFT) Spectrogram
The STFT magnitude spectrogram reveals how energy is distributed across
frequency and time. We use parameters aligned with the TinyML inference pipeline:
- FFT size: 512 points → 8 ms resolution at 16 kHz
- Hop length: 256 samples → 50% overlap

### 4.2 Mel-Scaled Spectrogram
Perceptually motivated frequency scale (40 Mel bands) to
match MFCC feature extraction used in the deployed model.

### 4.3 Constant-Q Transform (CQT)
Logarithmic frequency resolution captures harmonic structure
and transient onset characteristics better than linear STFT.

In [ ]:
fig = plt.figure(figsize=(16, 10))
outer = gridspec.GridSpec(len(CLASS_COLORS), 3, figure=fig,
                          hspace=0.45, wspace=0.35)

titles_top = ["STFT Spectrogram", "Mel Spectrogram (40 bands)", "CQT Spectrogram"]

for row, (label, color) in enumerate(CLASS_COLORS.items()):
    subset = df[df["label"] == label]
    if subset.empty:
        continue
    med_dur = subset["duration_s"].median()
    path = subset.iloc[(subset["duration_s"] - med_dur).abs().argsort().iloc[0]]["path"]
    y, sr = librosa.load(path, sr=SR_TARGET, mono=True, duration=4.0)

    specs = {
        "stft": np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)),
        "mel":  librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT,
                                               hop_length=HOP_LENGTH, n_mels=N_MELS),
        "cqt":  np.abs(librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH,
                                    bins_per_octave=12, n_bins=72)),
    }

    for col, (spec_name, S) in enumerate(specs.items()):
        ax = fig.add_subplot(outer[row, col])
        S_db = librosa.amplitude_to_db(S, ref=np.max)
        img = librosa.display.specshow(
            S_db, sr=sr, hop_length=HOP_LENGTH, x_axis="time",
            y_axis=("log" if spec_name == "stft" else
                    "mel" if spec_name == "mel" else "cqt_hz"),
            ax=ax, cmap="magma"
        )
        cbar = plt.colorbar(img, ax=ax, format="%+2.0f dB", pad=0.02)
        cbar.ax.tick_params(labelsize=8)
        row_label = f"[{label.upper()}] " if col == 0 else ""
        ax.set_title(f"{row_label}{titles_top[col]}", fontsize=11)
        ax.set_xlabel("Time (s)" if row == len(CLASS_COLORS)-1 else "")
        for sp in ax.spines.values():
            sp.set_linewidth(1.0)
        ax.tick_params(which="both", direction="in", top=False, right=False,
                       labelsize=9)

plt.suptitle("Spectral Representations Across Classes  (SR = 16 kHz, FFT = 512)",
             fontsize=13, fontweight="bold", y=1.01)
plt.savefig("figures/fig3_spectrograms.pdf")
plt.show()
print("Saved → figures/fig3_spectrograms.pdf")

## §5 — MFCC Analysis

MFCCs (Mel-Frequency Cepstral Coefficients) are the primary feature vector
fed into the TinyML classifier. This section analyses:

1. **MFCC heatmaps** — per-class coefficient evolution over time
2. **Mean MFCC profiles** — class-level spectral shape comparison  
3. **MFCC variance** — intra-class consistency (lower variance → more generalisable features)
4. **Delta & Delta-Delta MFCCs** — velocity and acceleration of spectral shape

In [ ]:
fig, axes = plt.subplots(len(CLASS_COLORS), 1, figsize=(14, 9))

for row, (label, color) in enumerate(CLASS_COLORS.items()):
    subset = df[df["label"] == label]
    if subset.empty:
        continue
    med_dur = subset["duration_s"].median()
    path = subset.iloc[(subset["duration_s"] - med_dur).abs().argsort().iloc[0]]["path"]
    y, sr = librosa.load(path, sr=SR_TARGET, mono=True, duration=4.0)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                  n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    ax = axes[row]
    img = librosa.display.specshow(mfcc, sr=sr, hop_length=HOP_LENGTH,
                                    x_axis="time", ax=ax, cmap="RdBu_r")
    cbar = plt.colorbar(img, ax=ax, pad=0.01)
    cbar.ax.tick_params(labelsize=8)
    cbar.set_label("Coefficient Value", fontsize=9)
    ax.set_ylabel(f"MFCC Index\n[{label.upper()}]")
    ax.set_yticks(range(N_MFCC))
    ax.set_yticklabels([f"C{i}" for i in range(N_MFCC)], fontsize=8)
    ax.set_title(f"({'abc'[row]}) MFCC Heatmap — {label.capitalize()}", fontsize=12)
    for sp in ax.spines.values():
        sp.set_linewidth(1.1)
    ax.tick_params(direction="in", top=True, right=True)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.savefig("figures/fig4_mfcc_heatmaps.pdf")
plt.show()
print("Saved → figures/fig4_mfcc_heatmaps.pdf")

In [ ]:
# ── Aggregate MFCCs for all files (this may take a few minutes) ──────────────
mfcc_means = defaultdict(list)
mfcc_stds  = defaultdict(list)

print("Extracting MFCCs...")
for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        y, sr = librosa.load(row["path"], sr=SR_TARGET, mono=True, duration=3.0)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        mfcc_means[row["label"]].append(np.mean(mfcc, axis=1))
        mfcc_stds[row["label"]].append(np.std(mfcc, axis=1))
    except:
        pass

mfcc_mean_df = {lbl: np.array(vals) for lbl, vals in mfcc_means.items()}
print("Done. Samples extracted per class:",
      {k: v.shape[0] for k, v in mfcc_mean_df.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
coeff_idx = np.arange(N_MFCC)

# ── A: Mean MFCC profile with std band ──────────────────────────────────────
ax = axes[0]
for label, color in CLASS_COLORS.items():
    if label not in mfcc_mean_df:
        continue
    mu  = mfcc_mean_df[label].mean(axis=0)
    std = mfcc_mean_df[label].std(axis=0)
    ax.plot(coeff_idx, mu, color=color, linewidth=1.8, marker="o",
            markersize=4, label=label.capitalize())
    ax.fill_between(coeff_idx, mu - std, mu + std, color=color, alpha=0.18)

ax.set_xlabel("MFCC Coefficient Index")
ax.set_ylabel("Mean Coefficient Value")
ax.set_title("(a) Mean MFCC Profile (±1σ band)")
ax.set_xticks(coeff_idx)
ax.set_xticklabels([f"C{i}" for i in coeff_idx], rotation=45, fontsize=9)
ax.legend()
paper_axes(ax)

# ── B: Intra-class variance heatmap ─────────────────────────────────────────
ax = axes[1]
labels_ordered = [l for l in CLASS_COLORS if l in mfcc_mean_df]
var_matrix = np.array([mfcc_mean_df[l].std(axis=0) for l in labels_ordered])
im = ax.imshow(var_matrix, aspect="auto", cmap="YlOrRd", interpolation="nearest")
cbar = plt.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("Std. Deviation", fontsize=10)
ax.set_xticks(coeff_idx)
ax.set_xticklabels([f"C{i}" for i in coeff_idx], rotation=45, fontsize=9)
ax.set_yticks(range(len(labels_ordered)))
ax.set_yticklabels([l.capitalize() for l in labels_ordered])
ax.set_xlabel("MFCC Coefficient Index")
ax.set_title("(b) Intra-Class MFCC Variance Heatmap")
for sp in ax.spines.values():
    sp.set_linewidth(1.1)
ax.tick_params(direction="in")

plt.tight_layout()
plt.savefig("figures/fig5_mfcc_profiles.pdf")
plt.show()
print("Saved → figures/fig5_mfcc_profiles.pdf")

## §6 — Handcrafted Acoustic Feature Extraction

Beyond MFCCs, the following spectral and temporal features are extracted
to quantify class-discriminative signal properties:

| Feature | Symbol | Physical Interpretation |
|---------|---------|------------------------|
| Spectral Centroid | SC | Brightness / centre of mass of spectrum |
| Spectral Bandwidth | SB | Spread of energy around centroid |
| Spectral Rolloff (85%) | SR | Frequency below which 85% of energy lies |
| Zero-Crossing Rate | ZCR | Noisiness / pitch roughness indicator |
| RMS Energy | RMS | Overall signal loudness |
| Spectral Flatness | SF | Tone-like vs. noise-like signal |
| Spectral Contrast | SCont | Peak-valley ratio per sub-band |

These features are used in §7 for PCA/t-SNE class separability analysis.

In [ ]:
feature_records = []

print("Extracting handcrafted features...")
for _, row_data in tqdm(df.iterrows(), total=len(df)):
    try:
        y, sr = librosa.load(row_data["path"], sr=SR_TARGET, mono=True, duration=3.0)
        if len(y) < N_FFT:
            continue

        sc    = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH))
        sb    = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH))
        sr85  = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85, n_fft=N_FFT, hop_length=HOP_LENGTH))
        zcr   = np.mean(librosa.feature.zero_crossing_rate(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH))
        rms   = np.mean(librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH))
        sf    = np.mean(librosa.feature.spectral_flatness(y=y, n_fft=N_FFT, hop_length=HOP_LENGTH))
        sc5   = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=N_FFT,
                                                   hop_length=HOP_LENGTH, n_bands=5)
        sc5_m = np.mean(sc5, axis=1)

        rec = {
            "label": row_data["label"],
            "spectral_centroid": sc,
            "spectral_bandwidth": sb,
            "spectral_rolloff85": sr85,
            "zcr": zcr,
            "rms_energy": rms,
            "spectral_flatness": sf,
        }
        for i, val in enumerate(sc5_m):
            rec[f"spectral_contrast_b{i}"] = val

        # MFCC means
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        for i in range(N_MFCC):
            rec[f"mfcc_mean_{i}"] = np.mean(mfcc[i])
            rec[f"mfcc_std_{i}"]  = np.std(mfcc[i])

        feature_records.append(rec)
    except Exception as e:
        pass

feat_df = pd.DataFrame(feature_records)
print(f"Feature matrix shape: {feat_df.shape}")
print(f"Classes: {feat_df['label'].value_counts().to_dict()}")
feat_df.head(3)

## §6.1 — Feature Distribution Analysis

Box + strip plots show the statistical distribution of each primary feature per class.
Separation between class distributions indicates discriminative power.

In [ ]:
primary_features = {
    "spectral_centroid":   "Spectral Centroid (Hz)",
    "spectral_bandwidth":  "Spectral Bandwidth (Hz)",
    "spectral_rolloff85":  "Spectral Rolloff 85% (Hz)",
    "zcr":                 "Zero-Crossing Rate",
    "rms_energy":          "RMS Energy",
    "spectral_flatness":   "Spectral Flatness",
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (feat, ylabel) in zip(axes.flatten(), primary_features.items()):
    classes  = [c for c in CLASS_COLORS if c in feat_df["label"].unique()]
    data     = [feat_df[feat_df["label"] == c][feat].dropna().values for c in classes]
    colors   = [CLASS_COLORS[c] for c in classes]

    bp = ax.boxplot(data, patch_artist=True, notch=True,
                    medianprops=dict(color="black", linewidth=2),
                    whiskerprops=dict(linewidth=1.0),
                    capprops=dict(linewidth=1.0),
                    flierprops=dict(marker="x", markersize=3, markeredgewidth=0.7, alpha=0.5))
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    # Strip overlay (jittered)
    for i, (d, color) in enumerate(zip(data, colors)):
        jitter = np.random.uniform(-0.18, 0.18, size=len(d))
        ax.scatter(np.full(len(d), i + 1) + jitter, d,
                   color=color, alpha=0.35, s=8, zorder=3, edgecolors="none")

    ax.set_xticks(range(1, len(classes) + 1))
    ax.set_xticklabels([c.capitalize() for c in classes], fontsize=10)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    paper_axes(ax)

plt.suptitle("Acoustic Feature Distributions per Class", fontsize=13,
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("figures/fig6_feature_distributions.pdf")
plt.show()
print("Saved → figures/fig6_feature_distributions.pdf")

## §6.2 — Statistical Summary Table

Descriptive statistics (mean ± std) for all primary features, formatted
for inclusion in the paper as **Table I**.

In [ ]:
summary_rows = []
for feat, ylabel in primary_features.items():
    for label in feat_df["label"].unique():
        vals = feat_df[feat_df["label"] == label][feat].dropna()
        summary_rows.append({
            "Feature":  ylabel,
            "Class":    label.capitalize(),
            "Mean":     round(vals.mean(), 5),
            "Std":      round(vals.std(),  5),
            "Median":   round(vals.median(), 5),
            "Min":      round(vals.min(),  5),
            "Max":      round(vals.max(),  5),
            "Skewness": round(skew(vals),     3),
            "Kurtosis": round(kurtosis(vals),  3),
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.set_index(["Feature", "Class"]).style.background_gradient(
    cmap="Blues", subset=["Mean"]).format(precision=4))

summary_df.to_csv("figures/table1_feature_statistics.csv", index=False)
print("Saved → figures/table1_feature_statistics.csv")

## §7 — Inter-Class Separability: PCA & t-SNE

Class separability directly predicts how well a TinyML classifier will perform.
- **PCA** (linear): reveals globally separated class clusters in the MFCC feature space
- **t-SNE** (non-linear): uncovers local neighbourhood structure and potential overlap regions

Feature vector used: 13 MFCC means + 13 MFCC stds + 6 primary spectral features = **32-dimensional vector**

In [ ]:
feature_cols = ([f"mfcc_mean_{i}" for i in range(N_MFCC)] +
                [f"mfcc_std_{i}"  for i in range(N_MFCC)] +
                list(primary_features.keys()))

feat_clean = feat_df[feature_cols + ["label"]].dropna()
X = feat_clean[feature_cols].values
y_labels = feat_clean["label"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── PCA ──────────────────────────────────────────────────────────────────────
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# ── t-SNE ────────────────────────────────────────────────────────────────────
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42, verbose=0)
X_tsne = tsne.fit_transform(X_scaled)

fig = plt.figure(figsize=(15, 5))

# ── A: PCA 2D ────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(131)
for label, color in CLASS_COLORS.items():
    mask = y_labels == label
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1],
                color=color, marker=CLASS_MARKERS[label],
                s=30, alpha=0.65, edgecolors="none", label=label.capitalize())
ax1.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax1.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax1.set_title("(a) PCA — PC1 vs PC2")
ax1.legend(markerscale=1.4)
paper_axes(ax1)

# ── B: PCA scree plot ────────────────────────────────────────────────────────
ax2 = fig.add_subplot(132)
pca_full = PCA(random_state=42).fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
n_plot = min(20, len(cumvar))
ax2.bar(range(1, n_plot+1), pca_full.explained_variance_ratio_[:n_plot]*100,
        color="#4393c3", edgecolor="black", linewidth=0.7, alpha=0.8, label="Individual")
ax2.plot(range(1, n_plot+1), cumvar[:n_plot], color="#d6604d",
         linewidth=1.8, marker="o", markersize=4, label="Cumulative")
ax2.axhline(90, color="black", linewidth=0.9, linestyle="--", label="90% threshold")
ax2.set_xlabel("Principal Component")
ax2.set_ylabel("Explained Variance (%)")
ax2.set_title("(b) PCA Scree Plot")
ax2.legend()
paper_axes(ax2)

# ── C: t-SNE ─────────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(133)
for label, color in CLASS_COLORS.items():
    mask = y_labels == label
    ax3.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                color=color, marker=CLASS_MARKERS[label],
                s=30, alpha=0.65, edgecolors="none", label=label.capitalize())
ax3.set_xlabel("t-SNE Dimension 1")
ax3.set_ylabel("t-SNE Dimension 2")
ax3.set_title("(c) t-SNE Embedding (perplexity=30)")
ax3.legend(markerscale=1.4)
paper_axes(ax3)

plt.tight_layout()
plt.savefig("figures/fig7_pca_tsne.pdf")
plt.show()
print("Saved → figures/fig7_pca_tsne.pdf")
print(f"Variance explained by PC1+PC2: {sum(pca.explained_variance_ratio_[:2])*100:.1f}%")

## §8 — Feature Correlation & Redundancy Analysis

A correlation heatmap of the 32-dimensional feature vector identifies:
- **Highly correlated pairs** (|r| > 0.85): redundant features that can be pruned to reduce model size
- **Near-zero correlations** with class label: low-discriminative features to deprioritise

This directly informs **feature selection** for the TinyML model to fit within ESP32 constraints.

In [ ]:
corr_cols = [f"mfcc_mean_{i}" for i in range(N_MFCC)] + list(primary_features.keys())
corr_labels = [f"C{i}" for i in range(N_MFCC)] + ["SC","SB","SR85","ZCR","RMS","SF"]

corr_matrix = feat_df[corr_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # upper triangle only

cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr_matrix,
            mask=mask,
            cmap=cmap,
            vmin=-1, vmax=1, center=0,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=0.4, linecolor="0.85",
            square=True, ax=ax,
            xticklabels=corr_labels,
            yticklabels=corr_labels,
            cbar_kws={"label": "Pearson r", "shrink": 0.8})

ax.set_title("Feature Correlation Matrix (MFCC Means + Spectral Features)",
             fontsize=13, fontweight="bold", pad=12)
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.tick_params(axis="y", rotation=0,  labelsize=9)

# Annotate highly correlated pairs
threshold = 0.85
for i in range(len(corr_matrix)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) >= threshold:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                        edgecolor="gold", lw=2.0))

from matplotlib.patches import Patch as MPatch
legend_elements = [MPatch(facecolor="none", edgecolor="gold", linewidth=2,
                           label=f"|r| ≥ {threshold} (high correlation)")]
ax.legend(handles=legend_elements, loc="upper right", fontsize=9)

plt.tight_layout()
plt.savefig("figures/fig8_correlation_heatmap.pdf")
plt.show()
print("Saved → figures/fig8_correlation_heatmap.pdf")

## §9 — Power Spectral Density (Welch Method) per Class

PSD characterises the frequency content of each class in steady state.
Key expected patterns:
- **Footsteps**: energy concentrated in 100–800 Hz (impact frequency)
- **Ballistic**: broadband impulsive energy across all frequencies
- **Noise**: predominantly low-frequency energy (< 500 Hz) with roll-off

This informs the frequency band passed to MFCC extraction.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for label, color in CLASS_COLORS.items():
    subset = df[df["label"] == label]
    if subset.empty:
        continue

    psd_accumulator = []
    for _, row_data in subset.sample(min(20, len(subset)), random_state=42).iterrows():
        try:
            y, sr = librosa.load(row_data["path"], sr=SR_TARGET, mono=True, duration=3.0)
            freqs, psd = welch(y, fs=sr, nperseg=512, noverlap=256)
            psd_accumulator.append(psd)
        except:
            pass

    if not psd_accumulator:
        continue

    psd_arr = np.array(psd_accumulator)
    psd_mean = 10 * np.log10(np.mean(psd_arr, axis=0) + 1e-12)
    psd_std  = 10 * np.log10(np.std(psd_arr, axis=0)  + 1e-12)

    ax.plot(freqs, psd_mean, color=color, linewidth=1.8, label=label.capitalize())
    ax.fill_between(freqs, psd_mean - 3, psd_mean + 3, color=color, alpha=0.15)

# Annotate footstep frequency band
ax.axvspan(100, 800, alpha=0.07, color="gray", label="Footstep band (100–800 Hz)")
ax.axvline(100, color="gray", linewidth=0.9, linestyle="--")
ax.axvline(800, color="gray", linewidth=0.9, linestyle="--")

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power Spectral Density (dB/Hz)")
ax.set_title("Mean Power Spectral Density per Class  (Welch, 20-sample average ±3 dB band)")
ax.set_xlim(0, SR_TARGET // 2)
ax.legend()
paper_axes(ax)

plt.tight_layout()
plt.savefig("figures/fig9_psd_comparison.pdf")
plt.show()
print("Saved → figures/fig9_psd_comparison.pdf")

## §10 — Dataset Quality Assessment

Before model training, the following quality checks are performed:

1. **Silence detection** — clips where RMS < threshold are likely corrupted or mislabelled
2. **Clipping detection** — samples with amplitude saturation (|amp| > 0.99) degrade MFCC quality
3. **Duration outliers** — files with unusual lengths may need trimming or padding
4. **Sample rate consistency** — all files should match the 16 kHz target

In [ ]:
quality_records = []

print("Running quality checks...")
for _, row_data in tqdm(df.iterrows(), total=len(df)):
    try:
        y, sr = librosa.load(row_data["path"], sr=None, mono=True)
        rms_val   = np.sqrt(np.mean(y**2))
        is_silent = rms_val < 0.001
        clip_ratio = np.mean(np.abs(y) > 0.99)
        is_clipped = clip_ratio > 0.01
        quality_records.append({
            "filename":    row_data["filename"],
            "label":       row_data["label"],
            "rms":         round(rms_val, 6),
            "clip_ratio":  round(clip_ratio, 5),
            "is_silent":   is_silent,
            "is_clipped":  is_clipped,
            "native_sr":   sr,
            "sr_mismatch": sr != SR_TARGET,
        })
    except:
        pass

qual_df = pd.DataFrame(quality_records)

print("\n── Quality Summary ──────────────────────────────────")
print(f"  Silent files   (RMS < 0.001) : {qual_df['is_silent'].sum()}")
print(f"  Clipped files  (>1% sat.)    : {qual_df['is_clipped'].sum()}")
print(f"  SR mismatch    (≠ 16 kHz)    : {qual_df['sr_mismatch'].sum()}")
print("─────────────────────────────────────────────────────")

display(qual_df.groupby("label")[["rms","clip_ratio","is_silent","is_clipped","sr_mismatch"]]\
        .agg({"rms":"mean","clip_ratio":"mean",
               "is_silent":"sum","is_clipped":"sum","sr_mismatch":"sum"}).round(5))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── A: RMS distribution per class ────────────────────────────────────────────
ax = axes[0]
for label, color in CLASS_COLORS.items():
    vals = qual_df[qual_df["label"] == label]["rms"].dropna()
    ax.hist(vals, bins=30, color=color, alpha=0.6, edgecolor="none",
            density=True, label=label.capitalize())
ax.axvline(0.001, color="red", linewidth=1.2, linestyle="--", label="Silence threshold")
ax.set_xlabel("RMS Energy")
ax.set_ylabel("Density")
ax.set_title("(a) RMS Energy Distribution")
ax.legend()
paper_axes(ax)

# ── B: Clipping ratio per class ──────────────────────────────────────────────
ax = axes[1]
for label, color in CLASS_COLORS.items():
    vals = qual_df[qual_df["label"] == label]["clip_ratio"].dropna() * 100
    ax.hist(vals, bins=30, color=color, alpha=0.6, edgecolor="none",
            density=True, label=label.capitalize())
ax.axvline(1.0, color="red", linewidth=1.2, linestyle="--", label="Clipping threshold (1%)")
ax.set_xlabel("Clipping Ratio (%)")
ax.set_ylabel("Density")
ax.set_title("(b) Signal Clipping Ratio")
ax.legend()
paper_axes(ax)

# ── C: Issue summary bar ─────────────────────────────────────────────────────
ax = axes[2]
issue_data = qual_df.groupby("label")[["is_silent","is_clipped","sr_mismatch"]].sum()
issue_data = issue_data.reindex([l for l in CLASS_COLORS if l in issue_data.index])
x = np.arange(len(issue_data))
w = 0.25
ax.bar(x - w,  issue_data["is_silent"],   width=w, label="Silent",   color="#4393c3", edgecolor="black", linewidth=0.7)
ax.bar(x,      issue_data["is_clipped"],  width=w, label="Clipped",  color="#d6604d", edgecolor="black", linewidth=0.7)
ax.bar(x + w,  issue_data["sr_mismatch"], width=w, label="SR Mismatch", color="#4dac26", edgecolor="black", linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels([l.capitalize() for l in issue_data.index])
ax.set_ylabel("File Count")
ax.set_title("(c) Quality Issues per Class")
ax.legend()
paper_axes(ax)

plt.tight_layout()
plt.savefig("figures/fig10_quality_assessment.pdf")
plt.show()
print("Saved → figures/fig10_quality_assessment.pdf")

## §11 — EDA Summary & Implications for Model Design

| Finding | Implication for TinyML Model |
|---------|------------------------------|
| Class imbalance (if detected) | Apply class weighting or augmentation (time-stretch, pitch-shift) |
| Footstep PSD peaks at 100–800 Hz | Band-pass pre-filter before MFCC extraction |
| MFCC C0–C4 carry most variance | 8–10 coefficients may suffice (reduces model input size) |
| t-SNE shows separable clusters | Linear/shallow CNN sufficient; deep models likely overkill for ESP32 |
| High-correlation MFCC pairs | PCA whitening or pruning before model input |
| Low clip ratio | Dataset is clean; no heavy pre-processing required |

### Next Steps
- **Notebook 02**: Data Preprocessing & Augmentation Pipeline  
- **Notebook 03**: TinyML Model Training (Edge Impulse / TFLite Micro)  
- **Notebook 04**: Model Quantisation, INT8 Conversion & ESP32 Deployment Analysis

> All figures saved to `./figures/` as high-resolution PDFs (300 DPI) for direct paper inclusion.

In [ ]:
import glob as gl

saved_figs = sorted(gl.glob("figures/*"))
print(f"{'File':<45} {'Size (KB)':>10}")
print("─" * 57)
for f in saved_figs:
    size_kb = os.path.getsize(f) / 1024
    print(f"{os.path.basename(f):<45} {size_kb:>9.1f} KB")

print(f"\n✅ EDA complete — {len(saved_figs)} figures/tables ready for paper.")